In [1]:
import numpy as np
import pandas as pd

# ---- Junction definition: EDIT to match Kalavasal Junction ----
ARMS  = ["Arm A (N)", "Arm B (E)", "Arm C (S)", "Arm D (W)"]   # replace with the real road names
GROUP = ["NS", "EW", "NS", "EW"]     # which demand pattern each arm follows (assumed)
DISCHARGE = 0.4                      # vehicles/second leaving an arm on green (assumed)
YELLOW = 3
FIXED_GREEN = 25                     # fixed-time green per phase (assumed, not observed)
MIN_GREEN, MAX_GREEN = 10, 60
PHASINGS = {"2-phase": [[0, 2], [1, 3]],
            "4-phase": [[0], [1], [2], [3]]}

def demand_per_hour(hour, group):
    m = np.exp(-0.5 * ((hour - 8) / 1.5) ** 2)
    e = np.exp(-0.5 * ((hour - 17) / 2) ** 2)
    per_5min = (8 + 35 * m + 30 * e) if group == "NS" else (6 + 25 * m + 40 * e)
    return per_5min * 12 / 2         # the two arms of a direction share its demand

def make_arrivals(seed, days=1):
    rng = np.random.default_rng(seed)
    t = np.arange(days * 86400)
    hour = (t / 3600) % 24
    return np.array([(rng.random(len(t)) < demand_per_hour(hour, g) / 3600).astype(int)
                     for g in GROUP])

def simulate(controller, arrivals, phases):
    n_arms, n = arrivals.shape
    q = np.zeros(n_arms, dtype=int)
    log = np.zeros((n_arms, n), dtype=int)
    credit = np.zeros(n_arms)
    p, state, timer, served = 0, "GREEN", 0, 0

    for t in range(n):
        q += arrivals[:, t]
        if state == "GREEN":
            for a in phases[p]:
                credit[a] += DISCHARGE
                if credit[a] >= 1:
                    if q[a] > 0:
                        q[a] -= 1
                        served += 1
                        credit[a] -= 1
                    else:
                        credit[a] = 0.0      # unused capacity is lost
        timer += 1
        if state == "GREEN" and controller({"phase_arms": phases[p], "timer": timer,
                                            "queues": q.copy()}):
            state, timer = "YELLOW", 0
        elif state == "YELLOW" and timer >= YELLOW:
            p, state, timer = (p + 1) % len(phases), "GREEN", 0
            credit[:] = 0
        log[:, t] = q

    hour = (np.arange(n) / 3600) % 24
    def delay(mask):
        return round(log[:, mask].sum() / max(arrivals[:, mask].sum(), 1), 2)

    out = {"delay_s_full_day": delay(np.ones(n, dtype=bool)),
           "delay_s_8am": delay((hour >= 8) & (hour < 9)),
           "delay_s_5pm": delay((hour >= 17) & (hour < 18))}
    for i, a in enumerate(ARMS):
        out[f"max_queue_{a}"] = int(log[i].max())
    out["arrived"], out["served"] = int(arrivals.sum()), int(served)
    return out

def fixed_ctrl(s):
    return s["timer"] >= FIXED_GREEN

def rule_ctrl(s):
    waiting = sum(s["queues"][a] for a in s["phase_arms"])
    return s["timer"] >= MAX_GREEN or (s["timer"] >= MIN_GREEN and waiting == 0)

def los(delay):
    for limit, grade in [(10, "A"), (20, "B"), (35, "C"), (55, "D"), (80, "E")]:
        if delay <= limit:
            return grade
    return "F"

arr = make_arrivals(1)
print("Vehicles per arm (whole day):", dict(zip(ARMS, arr.sum(axis=1))))
print("Vehicles per arm in the 8 AM hour:", dict(zip(ARMS, arr[:, 8*3600:9*3600].sum(axis=1))))

Vehicles per arm (whole day): {'Arm A (N)': np.int64(2831), 'Arm B (E)': np.int64(2578), 'Arm C (S)': np.int64(2852), 'Arm D (W)': np.int64(2618)}
Vehicles per arm in the 8 AM hour: {'Arm A (N)': np.int64(260), 'Arm B (E)': np.int64(166), 'Arm C (S)': np.int64(257), 'Arm D (W)': np.int64(184)}


In [2]:
rows = {}
for pname, ph in PHASINGS.items():
    for cname, ctrl in [("fixed_time", fixed_ctrl), ("rule_based", rule_ctrl)]:
        rows[(pname, cname)] = simulate(ctrl, arr, ph)

res = pd.DataFrame(rows).T
res["LOS_8am"] = res["delay_s_8am"].apply(los)
res["LOS_5pm"] = res["delay_s_5pm"].apply(los)
res

delay_s_full_day  delay_s_8am  delay_s_5pm  \
2-phase fixed_time             12.34        12.90        13.25   
        rule_based              7.70         8.18         8.38   
4-phase fixed_time             43.67        51.25        48.74   
        rule_based             22.61        24.35        26.36   

                    max_queue_Arm A (N)  max_queue_Arm B (E)  \
2-phase fixed_time                  6.0                  8.0   
        rule_based                  5.0                  6.0   
4-phase fixed_time                 14.0                 16.0   
        rule_based                 10.0                 11.0   

                    max_queue_Arm C (S)  max_queue_Arm D (W)  arrived  \
2-phase fixed_time                  7.0                  7.0  10879.0   
        rule_based                  5.0                  6.0  10879.0   
4-phase fixed_time                 17.0                 14.0  10879.0   
        rule_based                 10.0                  8.0  10879.0   

                     served LOS_8am LOS_5pm  
2-phase fixed_time  10877.0       B       B  
        rule_based  10878.0       A       A  
4-phase fixed_time  10877.0       D       D  
        rule_based  10878.0       C       C